# 进阶作业：韩语/俄语大模型 LoRA 微调  

**目标:** 本 Notebook 专门用于完成“将微调方法应用到其他语言”的进阶作业。我们将对一个韩语/俄语数据集进行 LoRA 微调，并对比不同超参数的性能。  

**核心路径:**  
- **工作目录:** `/home/mw/project/`  
- **原始数据:** `/home/mw/input/Russia39613961/part-677f75d865d8-000260.jsonl.gz`  
- **处理后数据:** `/home/mw/project/data/lora_training_data_korean.json`  
- **最佳模型:** `/home/mw/project/models/deepseek-lora-best-ko`

## 步骤 1: 环境设置与库安装

In [1]:
# 安装基础库和韩语处理库
!pip install transformers==4.41.2 datasets==2.19.0 peft==0.10.0 accelerate==0.30.1 torch==2.5.1 -i https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
!pip install rouge_chinese konlpy -i https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple

DEPRECATION: Loading egg at /opt/conda/lib/python3.11/site-packages/papermill-2.3.1-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
DEPRECATION: Loading egg at /opt/conda/lib/python3.11/site-packages/papermill-2.3.1-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple


In [2]:
# 导入核心模块并设置环境变量 (必须在导入transformers之前)
import os
import logging

# 设置Hugging Face国内镜像，这是解决网络问题的关键
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# 1c. 验证环境变量是否设置成功
# 在导入transformers等库之前，先确认环境变量
print(f"--- 验证环境变量 ---")
print(f"Hugging Face Endpoint 已设置为: {os.environ.get('HF_ENDPOINT')}")
print(f"--------------------")

# 1d. 导入其他模块
import json
import gzip
import re
import random
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from typing import List, Dict

from datasets import load_dataset, Dataset
# 此时导入transformers，它会读取上面设置好的环境变量
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from torch.optim import AdamW
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from konlpy.tag import Mecab
from rouge_chinese import Rouge

# 设置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# 创建项目目录
project_dir = "/home/mw/project"
os.makedirs(os.path.join(project_dir, "data"), exist_ok=True)
os.makedirs(os.path.join(project_dir, "models"), exist_ok=True)
logger.info(f"项目目录 '{project_dir}' 已设置。")

--- 验证环境变量 ---
Hugging Face Endpoint 已设置为: https://hf-mirror.com
--------------------


/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-25 01:25:36,970 - INFO - 项目目录 '/home/mw/project' 已设置。


## 步骤 2: 健壮的数据处理与高效加载

In [3]:
# ==============================================================================
# 步骤 2: 健壮的数据处理与高效加载 (已修复索引错误)
# ==============================================================================
import json
import gzip
import re
import os
from pathlib import Path
from tqdm import tqdm
import random
from typing import List, Dict
from datasets import load_dataset

def clean_text(text: str) -> str:
    if not text: return ""
    text = re.sub(r'\\n+', '\\n', text.strip())
    text = re.sub(r'\\s+', ' ', text)
    text = re.sub(r'<[^>]+>', '', text)
    return text.strip()

def robust_process_item(item: Dict, lang: str = 'ko') -> Dict:
    title, content = None, None
    if 'title' in item and 'content' in item:
        title, content = clean_text(item.get('title', '')), clean_text(item.get('content', ''))
    elif 'text' in item:
        text = clean_text(item['text'])
        lines = text.split('\\n', 1)
        title, content = (lines[0], lines[1]) if len(lines) > 1 else ("文章", text)
    else:
        string_values = sorted([v for v in item.values() if isinstance(v, str)], key=len, reverse=True)
        if len(string_values) >= 1:
            content = clean_text(string_values[0])
            title = clean_text(string_values[1]) if len(string_values) >= 2 else "文章"
    if not content or len(content) < 30: return None
    lang_map = {'ru': '俄语', 'ko': '韩语'}
    lang_name = lang_map.get(lang, '特定语言')
    return {"prompt": f"请根据以下标题生成一段带有专业术语的{lang_name}文本。\\n\\n标题: {title}", "output": content}

def process_raw_file_robust(input_path: str, output_path: str, lang: str = 'ko', max_samples: int = 5000):
    processed_data = []
    try:
        with gzip.open(input_path, 'rt', encoding='utf-8') as f:
            for line in tqdm(f, desc=f"正在从 {Path(input_path).name} 中提取样本"):
                try:
                    item = json.loads(line.strip())
                    if processed := robust_process_item(item, lang=lang):
                        processed_data.append(processed)
                    if len(processed_data) >= max_samples:
                        logger.info(f"已成功提取 {max_samples} 条样本，提前停止处理。")
                        break
                except (json.JSONDecodeError, Exception):
                    continue
    except Exception as e:
        logger.error(f"处理文件 {input_path} 时出错: {e}")
        return
    with open(output_path, 'w', encoding='utf-8') as f: json.dump(processed_data, f, ensure_ascii=False, indent=2)
    logger.info(f"数据采样处理完成！共提取了 {len(processed_data)} 条数据至: {output_path}")

# --- 关键修正：重写 load_and_prepare_data_hf 函数 ---
def load_and_prepare_data_hf(json_path, tokenizer, val_size=0.1):
    logger.info(f"使用 datasets.load_dataset 从 {json_path} 加载数据...")
    dataset = load_dataset("json", data_files=json_path, split="train")
    dataset = dataset.filter(lambda example: example.get("prompt") is not None and example.get("output") is not None)
    if not dataset: raise ValueError("过滤后未找到包含 'prompt' 和 'output' 的有效数据。")
    logger.info(f"数据过滤完成，保留 {len(dataset)} 条有效数据。")
    
    def format_and_tokenize(example):
        text = f"### Instruction:\\n{example['prompt']}\\n\\n### Response:\\n{example.get('output', '')}"
        return tokenizer(text, truncation=True, max_length=512)
        
    # 1. 映射分词函数，但 **不移除** 原始列
    tokenized_dataset = dataset.map(format_and_tokenize)
    
    # 2. 划分数据集
    split_dataset = tokenized_dataset.train_test_split(test_size=val_size, seed=42)
    train_dataset = split_dataset['train']
    val_dataset = split_dataset['test']
    
    # 3. 直接从 val_dataset (它现在包含 'prompt' 和 'output') 创建 prompts
    validation_prompts = [{"instruction": item["prompt"], "output": item["output"]} for item in val_dataset]
    
    # 4. 从返回的训练集中移除文本列，以便 DataLoader 正常工作
    columns_to_remove = [col for col in ["prompt", "output", "input"] if col in train_dataset.column_names]
    train_dataset = train_dataset.remove_columns(columns_to_remove)

    return train_dataset, val_dataset, validation_prompts


# --- 执行数据处理 ---
korean_raw_path = '/home/mw/input/Russia39613961/part-677f75d865d8-000260.jsonl.gz'
korean_processed_path = '/home/mw/project/data/lora_training_data_korean.json'
logger.info("处理韩语/俄语原始数据...")
process_raw_file_robust(korean_raw_path, korean_processed_path, lang='ko', max_samples=5000)

2025-07-25 01:25:36,986 - INFO - 处理韩语/俄语原始数据...
正在从 part-677f75d865d8-000260.jsonl.gz 中提取样本: 4363it [00:00, 14599.18it/s]2025-07-25 01:25:37,335 - INFO - 已成功提取 5000 条样本，提前停止处理。
正在从 part-677f75d865d8-000260.jsonl.gz 中提取样本: 4999it [00:00, 14472.52it/s]
2025-07-25 01:25:37,500 - INFO - 数据采样处理完成！共提取了 5000 条数据至: /home/mw/project/data/lora_training_data_korean.json


## 步骤 3: 定义韩语专用评估器

In [4]:
# ==============================================================================
# 步骤 3: 定义韩语/俄语专用评估器 (已修复 KeyError)
# ==============================================================================
class LanguageSpecificEvaluator:
    def __init__(self, tokenizer, device, lang='ko'):
        self.tokenizer = tokenizer
        self.device = device
        self.rouge = Rouge()
        self.lang = lang
        self.lang_tokenizer = None
        if self.lang == 'ko':
            try:
                self.lang_tokenizer = Mecab()
                logger.info("Mecab (韩语分词器) 初始化成功。")
            except Exception:
                logger.warning("Mecab 初始化失败，将回退到基于空格的分词。")

    def _get_rouge(self, reference: str, candidate: str) -> dict:
        if not candidate or not candidate.strip(): return {'rouge-1': 0.0, 'rouge-2': 0.0, 'rouge-l': 0.0}
        if self.lang_tokenizer:
            reference_cut = ' '.join(self.lang_tokenizer.morphs(reference))
            candidate_cut = ' '.join(self.lang_tokenizer.morphs(candidate))
        else:
            reference_cut, candidate_cut = reference, candidate
        try:
            scores = self.rouge.get_scores(candidate_cut, reference_cut)[0]
            return {'rouge-1': scores['rouge-1']['f'], 'rouge-2': scores['rouge-2']['f'], 'rouge-l': scores['rouge-l']['f']}
        except Exception as e:
            logger.error(f"计算ROUGE时出错: {e}")
            return {'rouge-1': 0.0, 'rouge-2': 0.0, 'rouge-l': 0.0}

    def evaluate_generation(self, model, validation_prompts: list) -> dict:
        model.eval()
        
        # --- 关键修正：将 item['prompt'] 修改为 item['instruction'] ---
        prompts = [f"### Instruction:\\n{item['instruction']}\\n\\n### Response:\\n" for item in validation_prompts]
        # -----------------------------------------------------------------
        
        references = [item['output'] for item in validation_prompts]
        inputs = self.tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(self.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7, top_p=0.95, pad_token_id=self.tokenizer.eos_token_id)
        generated_texts = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_rouge_scores = {'rouge-1': [], 'rouge-2': [], 'rouge-l': []}
        for gen_text, ref_text in zip(generated_texts, references):
            gen_output = gen_text.split("### Response:")[-1].strip()
            for k, v in self._get_rouge(ref_text, gen_output).items(): all_rouge_scores[k].append(v)
        return {f"avg_{k}": np.mean(v) for k, v in all_rouge_scores.items()}

## 步骤 4: 定义并执行韩语微调主函数

In [6]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def run_advanced_finetuning():
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
    data_path = "/home/mw/project/data/lora_training_data_korean.json"
    lang_code = 'ko' # 假设数据集为韩语

    # 在下载模型前再次确认环境变量，确保设置生效
    logger.info(f"即将从Hugging Face下载模型。确认镜像地址 HF_ENDPOINT: {os.environ.get('HF_ENDPOINT')}")
        
    logger.info(f"开始进阶作业：{lang_code} 语言微调任务")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    train_dataset, _, validation_prompts = load_and_prepare_data_hf(data_path, tokenizer)
    train_subset = train_dataset.select(range(min(400, len(train_dataset)))) # 防止数据集过小
    logger.info(f"使用训练子集进行实验，大小: {len(train_subset)}")

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    train_dataloader = DataLoader(train_subset, batch_size=4, shuffle=True, collate_fn=data_collator)

    lora_configs = [{"r": 8, "lora_alpha": 16}, {"r": 16, "lora_alpha": 32}]
    best_rouge_l, best_config = -1, None
    
    for config in lora_configs:
        logger.info(f"--- [{lang_code}] 测试配置: r={config['r']}, alpha={config['lora_alpha']} ---")
        base_model = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True, torch_dtype=torch.bfloat16).to(device)
        lora_config = LoraConfig(task_type=TaskType.CAUSAL_LM, lora_dropout=0.1, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], bias="none", **config)
        peft_model = get_peft_model(base_model, lora_config)
        optimizer = AdamW(peft_model.parameters(), lr=5e-5)

        for epoch in range(2):
            peft_model.train()
            for batch in tqdm(train_dataloader, desc=f"[{lang_code}] Epoch {epoch+1}"):
                outputs = peft_model(**{k: v.to(device) for k, v in batch.items()})
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

        logger.info(f"开始 {lang_code} 模型评估...")
        evaluator = LanguageSpecificEvaluator(tokenizer, device, lang=lang_code)
        rouge_metrics = evaluator.evaluate_generation(peft_model, validation_prompts[:50])
        for k, v in rouge_metrics.items(): logger.info(f"  - {k}: {v:.4f}")

        if rouge_metrics['avg_rouge-l'] > best_rouge_l:
            best_rouge_l, best_config = rouge_metrics['avg_rouge-l'], config
            logger.info(f"*** [{lang_code}] 发现新的最佳配置: {config} ***")
            best_model_path = f"/home/mw/project/models/deepseek-lora-best-{lang_code}"
            peft_model.save_pretrained(best_model_path)
            tokenizer.save_pretrained(best_model_path)
            logger.info(f"最佳 {lang_code} 模型已保存至: {best_model_path}")
        
        del base_model, peft_model, optimizer
        torch.cuda.empty_cache()

    logger.info(f"\n--- {lang_code} 微调任务完成 --- 最终最佳配置为: {best_config}")

# --- 执行进阶作业微调任务 ---
run_advanced_finetuning()

2025-07-25 01:26:40,754 - INFO - 即将从Hugging Face下载模型。确认镜像地址 HF_ENDPOINT: https://hf-mirror.com
2025-07-25 01:26:40,756 - INFO - 开始进阶作业：ko 语言微调任务
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
2025-07-25 01:26:41,438 - INFO - 使用 datasets.load_dataset 从 /home/mw/project/data/lora_training_data_korean.json 加载数据...
2025-07-25 01:26:42,152 - INFO - 数据过滤完成，保留 5000 条有效数据。
2025-07-25 01:26:42,478 - INFO - 使用训练子集进行实验，大小: 400
2025-07-25 01:26:42,478 - INFO - --- [ko] 测试配置: r=8, alpha=16 ---
[ko] Epoch 2: 100%|██████████| 100/100 [02:36<00:00,  1.57s/it]
2025-07-25 01:32:01,291 - INFO - 开始 ko 模型评估...
2025-07-25 01:32:01,297 - WARNING - Mecab 初始化失败，将回退到基于空格的分词。
2025-07-25 01:32:41,051 - INFO -   - avg_rouge-1: 0.0093
2025-07-25 01:32:41,052 - INFO -   - avg_rouge-2: 0.0021
2025-07-25 01:32:41,053 - INFO -   - avg_rouge-l: 0.0093
2025-07-25 01:32:41,054 - INFO - *** [ko] 发现新的最佳配置: {'r': 8, 'lora_alpha': 16} ***
/opt/conda/lib/py

## 步骤 5: 模型合并与最终测试

In [7]:
def merge_and_test_advanced():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    base_model_path = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
    lang_code = 'ko'
    lora_adapter_path = f"/home/mw/project/models/deepseek-lora-best-{lang_code}"
    merged_model_path = f"/home/mw/project/models/deepseek-merged-final-{lang_code}"

    if not os.path.exists(lora_adapter_path):
        logger.error(f"错误：找不到最佳模型 {lora_adapter_path}。请确保训练已成功。")
        return

    logger.info("开始加载基础模型用于合并...")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_path,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        device_map="auto"
    ).eval()

    logger.info(f"加载LoRA适配器从: {lora_adapter_path}")
    model_to_merge = PeftModel.from_pretrained(base_model, lora_adapter_path)
    merged_model = model_to_merge.merge_and_unload()
    logger.info("权重合并完成。")

    logger.info(f"保存合并后的模型至: {merged_model_path}")
    merged_model.save_pretrained(merged_model_path)
    tokenizer = AutoTokenizer.from_pretrained(lora_adapter_path)
    tokenizer.save_pretrained(merged_model_path)
    logger.info("模型和分词器已保存。")

    logger.info(f"\n--- 测试合并后的 {lang_code} 模型生成效果 ---")
    test_prompts = [
        "인공지능의 의미는 무엇인가요?", # "人工智能的意义是什么？"
        "재생 가능 에너지의 중요성에 대해 짧은 단락을 작성하십시오.", # "请写一段关于可再生能源重要性的短文。"
    ]
    
    for prompt in test_prompts:
        logger.info(f"\n测试提示: {prompt}")
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = merged_model.generate(**inputs, max_length=256, num_return_sequences=1, do_sample=True, temperature=0.7, top_p=0.95, pad_token_id=tokenizer.eos_token_id)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        logger.info(f"模型回答:\n{response}")
        logger.info("-" * 50)

# 执行模型合并与测试
merge_and_test_advanced()

2025-07-25 01:39:05,735 - INFO - 开始加载基础模型用于合并...
2025-07-25 01:39:08,267 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
2025-07-25 01:39:18,988 - INFO - 加载LoRA适配器从: /home/mw/project/models/deepseek-lora-best-ko
2025-07-25 01:39:19,294 - INFO - 权重合并完成。
2025-07-25 01:39:19,295 - INFO - 保存合并后的模型至: /home/mw/project/models/deepseek-merged-final-ko
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
2025-07-25 01:39:36,100 - INFO - 模型和分词器已保存。
2025-07-25 01:39:36,102 - INFO - 
--- 测试合并后的 ko 模型生成效果 ---
2025-07-25 01:39:36,102 - INFO - 
测试提示: 인공지능의 의미는 무엇인가요?
2025-07-25 01:39:46,002 - INFO - 模型回答:
인공지능의 의미는 무엇인가요? (지난 3일의 지속 information) 3일 전, 3일의 지속 information에 "인공지능의 의미는 무엇인가요?"라는 question을 올렸습니다. 3일 전에, 이 question을 올리기 위해 2일 전에 "인공지능의 의미는 무엇인가요?"라는 question을 올렸습니다. 2일 전에, 이 quest